# 1. Imports

In [91]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
import copy
import pickle
import unicodedata
from tqdm import tqdm

# 2. Methods

## 2.1 Loading Datasets

In [107]:
def get_dataset_dataframe(dataset_folder_name, dataset_sub_folder_name, dataset_file_name):
    # Move up from /notebooks → /Thesis
    base_dir = os.path.dirname(os.getcwd())

    # Full path to your .jsonl file
    file_path = os.path.join(base_dir, "datasets", dataset_folder_name, dataset_sub_folder_name, dataset_file_name)

    # Load with pandas (must use lines=True for .jsonl format)
    df = pd.read_json(file_path, lines=True)

    return df

## 2.2 Extract Golden Evidence

In [4]:
def extract_first_group(evidence_list):
    try:
        return evidence_list[0]  # first group: list of [anno_id, evid_id, page, sent_id]
    except:
        return []

## 2.3 Text Normalization

In [5]:
def normalize(text):
    return unicodedata.normalize("NFC", text) if isinstance(text, str) else text

## 2.4 Check Page Existence After Normalization

In [6]:
def norm_page_exists_in_df(df, target_page):
    norm_page = normalize(target_page)
    for group in df['golden_evidence_group']:
        if not group:
            continue
        for item in group:
            if normalize(item[2]) == norm_page:
                return True
    return False

## 2.5 Extract Evidence Text

In this, we 'll extract the evidence page name & sentence id from given evidence group, which is extracted from fever_df['golden_evidence_group'], which will be fed to the 'enriched_index' (created using fever_index & wiki_index contaning page name, setence id & sentence text) which will give us the required sentence text.

For one evidence group, all the extracted sentences will be joined as returned as a one string.

In [79]:
def extract_evidence_text(evidence_group, enriched_index):
    # Handle NOT ENOUGH INFO (null evidence)
    if not evidence_group or (isinstance(evidence_group[0], list) and evidence_group[0][2] is None):
        return ""
    
    texts = []
    for item in evidence_group:
        try:
            page_name = item[2]
            sentence_id = item[3]

            if page_name in enriched_index and sentence_id in enriched_index[page_name]:
                sentence = enriched_index[page_name][sentence_id]
                texts.append(sentence)
        except Exception as e:
            # Failsafe in case of bad structure
            print(f"Warning while processing evidence item: {item}, error: {e}")
            continue

    return " ".join(texts)

# 3. Preparing Dataset for Pre-processing

## 3.1 Loading fever dataset - Train, Test & Dev

In [4]:
fever_train_df = get_dataset_dataframe("FEVER", "raw", "train.jsonl")
fever_train_df.head()

Reading from: e:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\raw\train.jsonl


,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [..."
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271..."
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]"
3,156709,VERIFIABLE,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT VERIFIABLE,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


In [5]:
fever_train_df.shape

(145449, 5)

In [6]:
fever_paper_test_df = get_dataset_dataframe("FEVER", "raw", "paper_test.jsonl")
fever_paper_test_df.head()

Reading from: e:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\raw\paper_test.jsonl


,id,verifiable,label,claim,evidence
0,113501,NOT VERIFIABLE,NOT ENOUGH INFO,Grease had bad reviews.,"[[[133128, None, None, None]]]"
1,163803,VERIFIABLE,SUPPORTS,Ukrainian Soviet Socialist Republic was a foun...,"[[[296950, 288668, Ukrainian_Soviet_Socialist_..."
2,70041,VERIFIABLE,SUPPORTS,2 Hearts is a musical composition by Minogue.,"[[[225394, 230056, 2_Hearts_-LRB-Kylie_Minogue..."
3,202314,VERIFIABLE,REFUTES,The New Jersey Turnpike has zero shoulders.,"[[[238335, 240393, New_Jersey_Turnpike, 15]]]"
4,57085,NOT VERIFIABLE,NOT ENOUGH INFO,Legendary Entertainment is the owner of Wanda ...,"[[[178035, None, None, None], [182093, None, N..."


In [7]:
fever_paper_test_df.shape

(9999, 5)

In [8]:
fever_paper_dev_df = get_dataset_dataframe("FEVER", "raw", "paper_dev.jsonl")
fever_paper_dev_df.head()

Reading from: e:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\raw\paper_dev.jsonl


,id,verifiable,label,claim,evidence
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback...,"[[[108548, None, None, None]]]"
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]"
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0..."
3,166626,NOT VERIFIABLE,NOT ENOUGH INFO,Anne Rice was born in New Jersey.,"[[[191656, None, None, None], [191657, None, N..."
4,111897,VERIFIABLE,REFUTES,Telemundo is a English-language television net...,"[[[131371, 146144, Telemundo, 0]], [[131371, 1..."


In [9]:
fever_paper_dev_df.shape

(9999, 5)

## 3.2 Merging Fever Dataset

In [10]:
FEVER_df = pd.concat([fever_train_df, fever_paper_test_df, fever_paper_dev_df], ignore_index=True)
FEVER_df.head()

,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [..."
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271..."
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]"
3,156709,VERIFIABLE,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT VERIFIABLE,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


In [11]:
FEVER_df.shape

(165447, 5)

Verifying whether the dataset merged correctly or we missed anything.

Train + Test + Dev = 145449 + 9999 + 9999 = 165447

This is exactly what we have in the merged dataset. Therefore, merging is done successfully.

Data Description:
1. id: The ID of the claim
2. label: The annotated label for the claim - SUPPORTS|REFUTES|NOT ENOUGH INFO.
3. claim: The text of the claim.
4. evidence: 

    A list of evidence sets like this
    
    [
        [[Annotation ID, Evidence ID, Wikipedia URL/ Page Name, sentence ID] , ...], # evidence set 1
        [[Annotation ID, Evidence ID, Wikipedia URL/ Page Name, sentence ID] , ...], # evidence set 2
        ...
    ]
     
     or

    A [Annotation ID, Evidence ID, null, null] tuple if the label is NOT ENOUGH INFO.

5. verifiable: There are 2 kind of values - 'VERIFIABLE' & 'NOT VERIFIABLE'
    - Whenever a claim is verified ,i.e. either supports or refutes, with the given evidence, then, that comes under 'VERIFIABLE', whereas
    - When a claim can't be verified, i.e. neither supports nor refutes, in the absence or the appropriate evidence, then, that comes under 'NOT VERIFIABLE'.

## 3.3 Saving Merged Dataset

In [ ]:
output_path = os.path.join("..", "datasets", "FEVER", "pre-processed", "FEVER_Merged.jsonl")
FEVER_df.to_json(output_path, orient="records", lines=True)

# 4. Data Preprocessing

In [13]:
FEVER_df = get_dataset_dataframe("FEVER", "pre-processed", "FEVER_Merged.jsonl")
FEVER_df.head()

Reading from: e:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\pre-processed\FEVER_Merged.jsonl


,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [..."
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271..."
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]"
3,156709,VERIFIABLE,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT VERIFIABLE,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


## 4.1 Dropping irrelevant column - 'verifiable'

In [14]:
# Dropping the 'verifiable' column as we already have the 'label' column
FEVER_df = FEVER_df.drop(columns=['verifiable'])
FEVER_df.head()

,id,label,claim,evidence
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [..."
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271..."
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]"
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


## 4.2 Multi-evidence to Single-evidence Conversion

Here, we are converting the multi-evidence fever dataset to single-evidence by extracting the first evidence group and storing it as a golden evidence group.

In [15]:
FEVER_df['golden_evidence_group'] = FEVER_df['evidence'].apply(extract_first_group)
FEVER_df.head()

,id,label,claim,evidence,golden_evidence_group
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [...","[[92206, 104971, Nikolaj_Coster-Waldau, 7], [9..."
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271...","[[174271, 187498, Roman_Atwood, 1]]"
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]","[[255136, 254645, History_of_art, 2]]"
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]","[[180804, 193183, Adrienne_Bailon, 0]]"
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]","[[100277, None, None, None]]"


## 4.3 Formating 'golden_evidence_group' column

Let's explore this newly created column for all 3 label types:
1. 'SUPPORTS'
2. 'REFUTES'
3. 'NOT ENOUGH INFO'


In [16]:
FEVER_df[FEVER_df['label'] == 'SUPPORTS']['golden_evidence_group']

0         [[92206, 104971, Nikolaj_Coster-Waldau, 7], [9...
1                       [[174271, 187498, Roman_Atwood, 1]]
2                     [[255136, 254645, History_of_art, 2]]
5         [[151831, 166598, Homeland_-LRB-TV_series-RRB-...
8                       [[49158, 58489, Boston_Celtics, 3]]
                                ...                        
165432                 [[259746, 258294, Hanford_Site, 23]]
165434                     [[241367, 242917, Brad_Wilk, 4]]
165437                   [[83064, 94959, Reign_Over_Me, 1]]
165440                    [[21784, 26782, Miranda_Otto, 1]]
165441    [[138793, 153853, NAACP_Image_Award_for_Outsta...
Name: golden_evidence_group, Length: 86701, dtype: object

In [17]:
FEVER_df[FEVER_df['label'] == 'REFUTES']['golden_evidence_group']

3                    [[180804, 193183, Adrienne_Bailon, 0]]
14                   [[161295, 175782, Stranger_Things, 5]]
16                        [[89957, 102650, Puerto_Rico, 0]]
26        [[31205, 37902, Peggy_Sue_Got_Married, 0], [31...
27                        [[73660, 84912, Andy_Roddick, 7]]
                                ...                        
165412    [[169207, 182867, A_River_Runs_Through_It_-LRB...
165418                       [[160200, 174721, Arizona, 0]]
165420    [[88467, 101093, Winter's_Tale_-LRB-novel-RRB-...
165433            [[252357, 252156, Harvard_University, 8]]
165435      [[108293, 121833, Kerplunk_-LRB-album-RRB-, 2]]
Name: golden_evidence_group, Length: 36441, dtype: object

In [ ]:
FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO']['golden_evidence_group']

4                              [[100277, None, None, None]]
6                              [[173384, None, None, None]]
7                              [[273626, None, None, None]]
21                             [[160895, None, None, None]]
23                             [[248748, None, None, None]]
                                ...                        
165442    [[284225, None, None, None], [285168, None, No...
165443                          [[71922, None, None, None]]
165444                          [[49508, None, None, None]]
165445                         [[246624, None, None, None]]
165446                          [[92386, None, None, None]]
Name: golden_evidence_group, Length: 42305, dtype: object

Findings:
1. For 'SUPPORTS' & 'REFUTES', we saw that even in a single evidence group, we have multiple evidences which means when we extract all the sentences, we 'll merge those sentences into one evidence.
2. For 'NOT ENOUGH INFO', we saw that even in this we have multiple evidences which doesn't make sense and having just one evidence in the evidence group is sufficient as anyhow it's a null value.

Before processessing, let's verify whether any of the golden evidence group has any non-null values.

In [19]:
# Checking if any of the golden evidence group where claim = 'NOT ENOUGH INFO' has any non-null values

temp_df = FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO']['golden_evidence_group']

for group_list in temp_df:
    for each_group in group_list:
        if each_group[2] is not None or each_group[3] is not None: 
            print(each_group[2], each_group[3])

Since, nothing gets printed which means all the values are none. So, there is no point of keeping multiple evidences info and we should only keep the info of the first evidence only.

In [20]:
# Replacing the 'golden_evidence_group' for 'NOT ENOUGH INFO' label with the first evidence info only
for idx, row in FEVER_df.iterrows():
    if row['label'] == 'NOT ENOUGH INFO':
        FEVER_df.at[idx, 'golden_evidence_group'] = [row['golden_evidence_group'][0]]

In [21]:
FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO']['golden_evidence_group']

4         [[100277, None, None, None]]
6         [[173384, None, None, None]]
7         [[273626, None, None, None]]
21        [[160895, None, None, None]]
23        [[248748, None, None, None]]
                      ...             
165442    [[284225, None, None, None]]
165443     [[71922, None, None, None]]
165444     [[49508, None, None, None]]
165445    [[246624, None, None, None]]
165446     [[92386, None, None, None]]
Name: golden_evidence_group, Length: 42305, dtype: object

From the data at index '165442', it is clear that the multiple evidences are removed.

### 4.3.1 Saving the pre-processed FEVER dataset with golden evidence group

In [24]:
output_path = os.path.join("..", "datasets", "FEVER", "pre-processed", "FEVER_Merged_with_Golden_Evidence.jsonl")
FEVER_df.to_json(output_path, orient="records", lines=True)

## 4.4 Creating fever lookup dictionary


In [7]:
FEVER_df = get_dataset_dataframe("FEVER", "pre-processed", "FEVER_Merged_with_Golden_Evidence.jsonl")
FEVER_df.head()

Reading from: e:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\pre-processed\FEVER_Merged_with_Golden_Evidence.jsonl


,id,label,claim,evidence,golden_evidence_group
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [...","[[92206, 104971, Nikolaj_Coster-Waldau, 7], [9..."
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271...","[[174271, 187498, Roman_Atwood, 1]]"
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]","[[255136, 254645, History_of_art, 2]]"
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]","[[180804, 193183, Adrienne_Bailon, 0]]"
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]","[[100277, None, None, None]]"


Since, the structure of the 'golden_evidence_group' where claim is either 'SUPPORTS' or 'REFUTES' is like this 

[
    [Annotation ID, Evidence ID, Page Name, Sentence ID],
    [Annotation ID, Evidence ID, Page Name, Sentence ID],
    [Annotation ID, Evidence ID, Page Name, Sentence ID],
    .
    .
    .
    [Annotation ID, Evidence ID, Page Name, Sentence ID]
]

Since, this is for one claim, so there 'll mutliple sentences (evidences) from the same page (or with same page name) and they might be spread over the entire data which means same data is being repeated multiple time.

Since, we need to find the corresponding sentences from wiki dump data so it is better to create a look-up dictionary. In this, all the 'Sentence ID' belongs to one page will come under one page name at one place. 

Later on, we 'll use this dictionary to add the sentence along with it which makes the lookup faster.

In [8]:
FEVER_df['golden_evidence_group'].head()

0    [[92206, 104971, Nikolaj_Coster-Waldau, 7], [9...
1                  [[174271, 187498, Roman_Atwood, 1]]
2                [[255136, 254645, History_of_art, 2]]
3               [[180804, 193183, Adrienne_Bailon, 0]]
4                         [[100277, None, None, None]]
Name: golden_evidence_group, dtype: object

In [30]:
fever_lookup = defaultdict(set)

for _, row in FEVER_df.iterrows():
    group = row['golden_evidence_group']
    if not group:
        continue
    for list_item in group:
        page, sent_id = list_item[2], list_item[3]
        if page and sent_id is not None:
            fever_lookup[page].add(sent_id)

fever_lookup

defaultdict(set,
            {'Nikolaj_Coster-Waldau': {0, 2, 3, 7, 8},
             'Fox_Broadcasting_Company': {0, 1, 2},
             'Roman_Atwood': {0, 1},
             'History_of_art': {0, 2},
             'Adrienne_Bailon': {0, 1, 8},
             'Homeland_-LRB-TV_series-RRB-': {0, 3},
             'Prisoners_of_War_-LRB-TV_series-RRB-': {0},
             'Boston_Celtics': {0, 3},
             'The_Ten_Commandments_-LRB-1956_film-RRB-': {0},
             'Tetris': {0, 18},
             'Cyndi_Lauper': {0, 2, 7, 21, 26},
             'The_Hunger_Games_-LRB-film-RRB-': {0,
              1,
              2,
              3,
              4,
              8,
              16,
              18,
              19,
              21,
              24,
              27},
             'Ryan_Gosling': {0, 3, 6, 7, 8, 9, 12, 17},
             'Chad': {0},
             'Stranger_Things': {0, 1, 2, 5, 6, 9, 13, 14, 16},
             'Ryan_Seacrest': {0},
             'Puerto_Rico': {0, 3, 6}

### 4.4.1 Saving this dictionary

In [31]:
# Define path to save the pickle file
output_path = r"E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\pre-processed\fever_lookup.pkl"

# Save the dictionary
with open(output_path, "wb") as f:
    pickle.dump(fever_lookup, f)

print(f"fever_lookup saved to: {output_path}")

fever_lookup saved to: E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\pre-processed\fever_lookup.pkl


## 4.5 Creating 'wiki_index' file

For creating this, we 'll using wiki dump files. There are total 109 files in the JSONL format.

We have formated & converted these JSONL files to pickle file to get the data in required structure and for faster processing.

Raw format is like this: {"id": "", "text": "", "lines": ""}

Example:

{
 "id": "1928_in_association_football",
 "text": "The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world . ",
 "lines": "0\tThe following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world .\n1\t"
}

Formatted pickle file structure: {'page name': 
                                    {
                                        <sentence_id>: <sentence text string>,
                                        <sentence_id>: <sentence text string>,
                                        <sentence_id>: <sentence text string>,
                                        .
                                        .
                                        .
                                        <sentence_id>: <sentence text string>,
                                    },

                                 'page name': 
                                    {
                                        <sentence_id>: <sentence text string>,
                                        <sentence_id>: <sentence text string>,
                                        <sentence_id>: <sentence text string>,
                                        .
                                        .
                                        .
                                        <sentence_id>: <sentence text string>,
                                    } 
                                 }

Example:

defaultdict(dict,
            {'1928_in_association_football':
                {
                    0: 'The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world .'
                },
             '1986_NBA_Finals': 
                {
                    0: 'The 1986 NBA Finals was the championship round of the 1985 -- 86 NBA season .\tNBA\tNational Basketball Association',

                    1: "It pitted the Eastern Conference champion Boston Celtics against the Western Conference champion Houston Rockets , in a rematch of the 1981 Finals -LRB- only Allen Leavell and Robert Reid remained from the Rockets ' 1981 team -RRB- .\tHouston Rockets\tHouston Rockets\tBoston Celtics\tBoston Celtics\tEastern Conference\tEastern Conference (NBA)\tWestern Conference\tWestern Conference (NBA)\tAllen Leavell\tAllen Leavell\tRobert Reid\tRobert Reid (basketball)\tRockets\tHouston Rockets\tCeltics\tBoston Celtics",

                    2: 'The Celtics defeated the Rockets four games to two to win their 16th NBA championship .\tNBA\tNational Basketball Association\tRockets\tHouston Rockets\tCeltics\tBoston Celtics',

                    3: "The championship...
                }
            })

There are data in this format in 109 pickle files. Now, the sentence we are looking for might not be present in one file so to find a sentence, in the worst case scenario, we might need to go through all the files.

Doing this for finding each sentence for fever_loopup is very computationally heavy. So, for this as well, we 'll create a wiki lookup or wiki index of all these 109 files which have the following format.

dict{
    <page name>:
        {
            sentence_id: <sentence text string>,
            sentence_id: <sentence text string>,
            .
            .
            .
            sentence_id: <sentence text string>,
        },
        .
        .
        .
    <page name>:
        {
            sentence_id: <sentence text string>,
            sentence_id: <sentence text string>,
            .
            .
            .
            sentence_id: <sentence text string>,
        }
}

Example:
defaultdict(dict,
            {'Fredvang_Bridges': {0: 'The Fredvang Bridges -LRB- Fredvangbruene -RRB- are two cantilever bridges that connect the fishing village of Fredvang on Moskenesøya island with the neighboring island of Flakstadøya .\tcantilever bridges\tcantilever bridges\tfishing village\tfishing village\tFredvang\tFredvang\tMoskenesøya\tMoskenesøya\tFlakstadøya\tFlakstadøya',
              1: 'The bridges are located in the municipality of Flakstad in Nordland county , Norway .\tFlakstad\tFlakstad\tNorway\tNorway\tNordland\tNordland',
              2: '',
              3: '',
              4: 'The south bridge is called Kubholmleia Bridge -LRB- Kubholmleia bru -RRB- and the north bridge is Røssøystraumen Bridge -LRB- Røssøystraumen bru -RRB- .',
              5: 'Both bridges are 240 m long and have a main span of 115 m.',

I formatted and converted the jsonl to pickle file using a python script 'wiki_jsonl_to_pkl.py'.

Because of the computational constrating, I created 'wiki_index' on kaggle notebook using the below code.

In [ ]:
'''
wiki_pickle_dir = r"E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\Wiki-Dump\pickle_file"
wiki_index = defaultdict(dict)

for file_name in os.listdir(wiki_pickle_dir):
    if not file_name.endswith(".pkl"):
        continue

    file_path = os.path.join(wiki_pickle_dir, file_name)
    with open(file_path, "rb") as f:
        page_dict = pickle.load(f)

    for page, sentences in page_dict.items():
        for sent_id, text in sentences.items():
            if sent_id not in wiki_index[page]:
                wiki_index[page][sent_id] = text
            elif wiki_index[page][sent_id] != text:
                print(f"Warning: Conflicting text for {page} at sentence {sent_id}")
'''

## 4.6 Loading 'wiki_index' file

In [13]:
with open(r"E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\Wiki-Dump\processed\old_wiki_index(with_metadata).pkl", "rb") as f:
    wiki_index = pickle.load(f)

wiki_index

defaultdict(dict,
            {'Fredvang_Bridges': {0: 'The Fredvang Bridges -LRB- Fredvangbruene -RRB- are two cantilever bridges that connect the fishing village of Fredvang on Moskenesøya island with the neighboring island of Flakstadøya .\tcantilever bridges\tcantilever bridges\tfishing village\tfishing village\tFredvang\tFredvang\tMoskenesøya\tMoskenesøya\tFlakstadøya\tFlakstadøya',
              1: 'The bridges are located in the municipality of Flakstad in Nordland county , Norway .\tFlakstad\tFlakstad\tNorway\tNorway\tNordland\tNordland',
              2: '',
              3: '',
              4: 'The south bridge is called Kubholmleia Bridge -LRB- Kubholmleia bru -RRB- and the north bridge is Røssøystraumen Bridge -LRB- Røssøystraumen bru -RRB- .',
              5: 'Both bridges are 240 m long and have a main span of 115 m.',
              6: '',
              7: '',
              8: 'The bridges were opened in 1988 .',
              9: 'They are among many bridges that connec

In [14]:
wiki_index['Nikolaj_Coster-Waldau']

{0: 'Nikolaj Coster-Waldau -LRB- -LSB- neɡ̊olaɪ̯ kʰʌsd̥ɐ ˈʋald̥ɑʊ̯ -RSB- ; born 27 July 1970 -RRB- is a Danish actor , producer and screenwriter .',
 1: 'He graduated from Danish National School of Theatre in Copenhagen in 1993 .\tDanish National School of Theatre\tDanish National School of Theatre and Contemporary Dance\tCopenhagen\tCopenhagen',
 2: "Coster-Waldau 's breakthrough performance in Denmark was his role in the film Nightwatch -LRB- 1994 -RRB- .\tNightwatch\tNightwatch (1994 film)",
 3: 'Since then he has appeared in numerous films in his native Scandinavia and Europe in general , including Headhunters -LRB- 2011 -RRB- and A Thousand Times Good Night -LRB- 2013 -RRB- .\tHeadhunters\tHeadhunters (film)\tA Thousand Times Good Night\tA Thousand Times Good Night',
 4: '',
 5: '',
 6: 'In the United States , his debut film role was in the war film Black Hawk Down -LRB- 2001 -RRB- , playing Medal of Honor recipient Gary Gordon .\tBlack Hawk Down\tBlack Hawk Down (film)\tGary Gord

In [15]:
wiki_index['Fredvang_Bridges']

{0: 'The Fredvang Bridges -LRB- Fredvangbruene -RRB- are two cantilever bridges that connect the fishing village of Fredvang on Moskenesøya island with the neighboring island of Flakstadøya .\tcantilever bridges\tcantilever bridges\tfishing village\tfishing village\tFredvang\tFredvang\tMoskenesøya\tMoskenesøya\tFlakstadøya\tFlakstadøya',
 1: 'The bridges are located in the municipality of Flakstad in Nordland county , Norway .\tFlakstad\tFlakstad\tNorway\tNorway\tNordland\tNordland',
 2: '',
 3: '',
 4: 'The south bridge is called Kubholmleia Bridge -LRB- Kubholmleia bru -RRB- and the north bridge is Røssøystraumen Bridge -LRB- Røssøystraumen bru -RRB- .',
 5: 'Both bridges are 240 m long and have a main span of 115 m.',
 6: '',
 7: '',
 8: 'The bridges were opened in 1988 .',
 9: 'They are among many bridges that connect the islands of the Lofoten archipelago to each other .\tLofoten\tLofoten\tarchipelago\tarchipelago',
 10: 'The only other bridge connecting these two islands is the K

In [16]:
wiki_index['Fredvang_Bridges'][0]

'The Fredvang Bridges -LRB- Fredvangbruene -RRB- are two cantilever bridges that connect the fishing village of Fredvang on Moskenesøya island with the neighboring island of Flakstadøya .\tcantilever bridges\tcantilever bridges\tfishing village\tfishing village\tFredvang\tFredvang\tMoskenesøya\tMoskenesøya\tFlakstadøya\tFlakstadøya'

We may notice that the sentence text is not entirely the text. The structure is something like this

<sentence_text> \t <mention1> \t <entity1> \t <mention2> \t <entity2> ...

This mention-entity is the sentence metadata which we don't need. Therefore, we 'll remove the metadata and just keep the sentence.

### 4.6.1 Removing Metadata from 'wiki_index' Sentences

In [17]:
# Removing Metadata from the sentences
for page, sentences in wiki_index.items():
    for sent_id, full_text in sentences.items():
        if full_text:
            wiki_index[page][sent_id] = full_text.split('\t')[0].strip()

In [18]:
wiki_index['Fredvang_Bridges'][0]

'The Fredvang Bridges -LRB- Fredvangbruene -RRB- are two cantilever bridges that connect the fishing village of Fredvang on Moskenesøya island with the neighboring island of Flakstadøya .'

In [19]:
# Saving the cleaned version
with open(r"E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\Wiki-Dump\processed\wiki_index.pkl", "wb") as f:
    pickle.dump(wiki_index, f)

print("Overwriting the existing file with this cleaned version")

Overwriting the existing file with this cleaned version


## 4.7 Creating enriched_fever_index

Now, we have both the required files - fever_lookup & wiki_index to extract the required sentence and create a new dictionary 'enriched_fever_index' having the following structure

{
    <page name string>: [
        (<sentence_id>, <sentence_text_string>),
        (<sentence_id>, <sentence_text_string>),
        .
        .
        .
        (<sentence_id>, <sentence_text_string>),
    ],
    .
    .
    .
    <page name string>: [
        (<sentence_id>, <sentence_text_string>),
        (<sentence_id>, <sentence_text_string>),
        .
        .
        .
        (<sentence_id>, <sentence_text_string>),
    ],
}

To create this dictionary, we 'll extract the 'page name' & 'sentence id' from fever_index (loaded variable of fever_lookup) and use those to find the sentence from wiki_inedx. 

With all this information, we 'll create the above a dictionary as stated above.

In [20]:
# Loading the fever_lookup dictionary from the pickle file

with open(r"E:\Users\Anirudh\Study\MS\Implementation\Thesis\datasets\FEVER\pre-processed\fever_lookup.pkl", "rb") as f:
    fever_index = pickle.load(f)

fever_index

defaultdict(set,
            {'Nikolaj_Coster-Waldau': {0, 2, 3, 7, 8},
             'Fox_Broadcasting_Company': {0, 1, 2},
             'Roman_Atwood': {0, 1},
             'History_of_art': {0, 2},
             'Adrienne_Bailon': {0, 1, 8},
             'Homeland_-LRB-TV_series-RRB-': {0, 3},
             'Prisoners_of_War_-LRB-TV_series-RRB-': {0},
             'Boston_Celtics': {0, 3},
             'The_Ten_Commandments_-LRB-1956_film-RRB-': {0},
             'Tetris': {0, 18},
             'Cyndi_Lauper': {0, 2, 7, 21, 26},
             'The_Hunger_Games_-LRB-film-RRB-': {0,
              1,
              2,
              3,
              4,
              8,
              16,
              18,
              19,
              21,
              24,
              27},
             'Ryan_Gosling': {0, 3, 6, 7, 8, 9, 12, 17},
             'Chad': {0},
             'Stranger_Things': {0, 1, 2, 5, 6, 9, 13, 14, 16},
             'Ryan_Seacrest': {0},
             'Puerto_Rico': {0, 3, 6}

In [21]:
enriched_fever_index = defaultdict(list)

# count how many lookups fail
missing_sentences = 0

for page, sentence_ids in fever_index.items():
    for sentence_id in sentence_ids:
        try:
            sent_text = wiki_index[page][sentence_id]
            enriched_fever_index[page].append((sentence_id, sent_text))
        except KeyError:
            enriched_fever_index[page].append((sentence_id, None))
            missing_sentences += 1
        
print(f"Enrichment complete. {missing_sentences} sentence(s) were not found.")

Enrichment complete. 165 sentence(s) were not found.


In [22]:
enriched_fever_index

defaultdict(list,
            {'Nikolaj_Coster-Waldau': [(0,
               'Nikolaj Coster-Waldau -LRB- -LSB- neɡ̊olaɪ̯ kʰʌsd̥ɐ ˈʋald̥ɑʊ̯ -RSB- ; born 27 July 1970 -RRB- is a Danish actor , producer and screenwriter .'),
              (2,
               "Coster-Waldau 's breakthrough performance in Denmark was his role in the film Nightwatch -LRB- 1994 -RRB- ."),
              (3,
               'Since then he has appeared in numerous films in his native Scandinavia and Europe in general , including Headhunters -LRB- 2011 -RRB- and A Thousand Times Good Night -LRB- 2013 -RRB- .'),
              (7,
               'He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot .'),
              (8,
               'He became widely known to a broad audience for his current role as Ser Jaime Lannister , in the HBO series Game o

### 4.7.1 Inspect Missing Entries

In [23]:
missing_entries = []

for page, entries in enriched_fever_index.items():
    for sent_id, sentence in entries:
        if sentence is None:
            missing_entries.append((page, sent_id))

# Show first few missing entries
for entry in missing_entries[:10]:
    print(f"Missing: Page = {entry[0]}, Sentence ID = {entry[1]}")

print(f"\nTotal entries with missing sentence: {len(missing_entries)}")

Missing: Page = José_María_Chacón, Sentence ID = 0
Missing: Page = José_María_Chacón, Sentence ID = 1
Missing: Page = Régine_Chassagne, Sentence ID = 0
Missing: Page = Régine_Chassagne, Sentence ID = 1
Missing: Page = Chris_Pérez, Sentence ID = 0
Missing: Page = Chris_Pérez, Sentence ID = 1
Missing: Page = Chris_Pérez, Sentence ID = 2
Missing: Page = Chris_Pérez, Sentence ID = 4
Missing: Page = Chris_Pérez, Sentence ID = 5
Missing: Page = Chris_Pérez, Sentence ID = 6

Total entries with missing sentence: 165


In [24]:
missing_entries

[('José_María_Chacón', 0),
 ('José_María_Chacón', 1),
 ('Régine_Chassagne', 0),
 ('Régine_Chassagne', 1),
 ('Chris_Pérez', 0),
 ('Chris_Pérez', 1),
 ('Chris_Pérez', 2),
 ('Chris_Pérez', 4),
 ('Chris_Pérez', 5),
 ('Chris_Pérez', 6),
 ('Beyoncé', 0),
 ('Beyoncé', 1),
 ('Beyoncé', 3),
 ('Beyoncé', 6),
 ('Beyoncé', 7),
 ('Beyoncé', 16),
 ('Beyoncé', 17),
 ('Beyoncé', 18),
 ('Beyoncé', 19),
 ('Beyoncé', 20),
 ('Frédéric_Auguste_Bartholdi', 0),
 ('Björn_Borg', 0),
 ('Citadelle_Laferrière', 0),
 ('Édith_Piaf', 0),
 ('Peer_Åström', 0),
 ('Peer_Åström', 1),
 ('Peer_Åström', 2),
 ('Peer_Åström', 8),
 ('Peer_Åström', 13),
 ('Christine_Daaé', 0),
 ('Ólafur_Arnalds', 0),
 ('Thérèse_Raquin', 0),
 ('Henri_Poincaré', 0),
 ('Daniela_Hantuchová', 0),
 ('Daniela_Hantuchová', 1),
 ('Daniela_Hantuchová', 9),
 ('2011_Vuelta_a_España', 16),
 ('Sofía_Vergara', 0),
 ('Sofía_Vergara', 11),
 ('Sofía_Vergara', 4),
 ('José_Aldo', 0),
 ('José_Aldo', 1),
 ('José_

This means that these 'page name' exist in fever_index but there is no match in the wiki_index.

Since, the fever_index is created out of the golden evidences of the fever dataset. Therefore, I should be able to find these in the 'FEVER_Merged_with_Golden_Evidence.jsonl' but when I try finding manually using 'ctrl + f', there is no result.

This might be because the names like 'José_María_Chacón' which includes characters like é, í, etc need to be normalized because "José_María_Chacón" != "José_María_Chacón" when compared byte-for-byte, even though they look the same because of the way they stored - NGD or NFC. 

Therefore, let's normalize the pages before we access them.

### 4.7.2 'Page Name' Normalization

In [25]:
fever_index

defaultdict(set,
            {'Nikolaj_Coster-Waldau': {0, 2, 3, 7, 8},
             'Fox_Broadcasting_Company': {0, 1, 2},
             'Roman_Atwood': {0, 1},
             'History_of_art': {0, 2},
             'Adrienne_Bailon': {0, 1, 8},
             'Homeland_-LRB-TV_series-RRB-': {0, 3},
             'Prisoners_of_War_-LRB-TV_series-RRB-': {0},
             'Boston_Celtics': {0, 3},
             'The_Ten_Commandments_-LRB-1956_film-RRB-': {0},
             'Tetris': {0, 18},
             'Cyndi_Lauper': {0, 2, 7, 21, 26},
             'The_Hunger_Games_-LRB-film-RRB-': {0,
              1,
              2,
              3,
              4,
              8,
              16,
              18,
              19,
              21,
              24,
              27},
             'Ryan_Gosling': {0, 3, 6, 7, 8, 9, 12, 17},
             'Chad': {0},
             'Stranger_Things': {0, 1, 2, 5, 6, 9, 13, 14, 16},
             'Ryan_Seacrest': {0},
             'Puerto_Rico': {0, 3, 6}

In [26]:
# Normalize fever_index keys
normalized_fever_index = defaultdict(set)
for page, sent_id_set in fever_index.items():
    norm_page = normalize(page)
    normalized_fever_index[norm_page].update(sent_id_set)

# Normalize wiki_index keys
normalized_wiki_index = defaultdict(dict)
for page, sent_dict in wiki_index.items():
    norm_page = normalize(page)
    normalized_wiki_index[norm_page].update(sent_dict)

Now, we 'll try creating enriched index again and see the no. of missing pages now.

In [ ]:
# enriched_fever_index = defaultdict(list)

# # count how many lookups fail
# missing_sentences = 0

# for page, sentence_ids in normalized_fever_index.items():
#     for sentence_id in sentence_ids:
#         try:
#             sent_text = normalized_wiki_index[page][sentence_id]
#             enriched_fever_index[page].append((sentence_id, sent_text))
#         except KeyError:
#             enriched_fever_index[page].append((sentence_id, None))
#             missing_sentences += 1
        
# print(f"Enrichment complete. {missing_sentences} sentence(s) were not found.")

Enrichment complete. 0 sentence(s) were not found.


In [74]:
enriched_fever_index = defaultdict(dict)
missing_sentences = 0

for page, sentence_ids in fever_index.items():
    for sentence_id in sentence_ids:
        try:
            sent_text = wiki_index[page][sentence_id]
            enriched_fever_index[page][sentence_id] = sent_text
        except KeyError:
            enriched_fever_index[page][sentence_id] = None
            missing_sentences += 1

print(f"Enrichment complete. {missing_sentences} sentence(s) were not found.")

Enrichment complete. 0 sentence(s) were not found.


As there are no missing sentences, therefore, all the names have been successfully normalized. So, now we can safely replace old index with the new index

In [75]:
# Replacing the original fever_index and wiki_index with the normalized versions
fever_index = normalized_fever_index
wiki_index = normalized_wiki_index

In [76]:
enriched_fever_index

defaultdict(dict,
            {'Nikolaj_Coster-Waldau': {0: 'Nikolaj Coster-Waldau -LRB- -LSB- neɡ̊olaɪ̯ kʰʌsd̥ɐ ˈʋald̥ɑʊ̯ -RSB- ; born 27 July 1970 -RRB- is a Danish actor , producer and screenwriter .',
              2: "Coster-Waldau 's breakthrough performance in Denmark was his role in the film Nightwatch -LRB- 1994 -RRB- .",
              3: 'Since then he has appeared in numerous films in his native Scandinavia and Europe in general , including Headhunters -LRB- 2011 -RRB- and A Thousand Times Good Night -LRB- 2013 -RRB- .',
              7: 'He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot .',
              8: 'He became widely known to a broad audience for his current role as Ser Jaime Lannister , in the HBO series Game of Thrones .'},
             'Fox_Broadcasting_Company': {0: 'The Fox Broadcasting Co

In [29]:
enriched_fever_index

defaultdict(list,
            {'Nikolaj_Coster-Waldau': [(0,
               'Nikolaj Coster-Waldau -LRB- -LSB- neɡ̊olaɪ̯ kʰʌsd̥ɐ ˈʋald̥ɑʊ̯ -RSB- ; born 27 July 1970 -RRB- is a Danish actor , producer and screenwriter .'),
              (2,
               "Coster-Waldau 's breakthrough performance in Denmark was his role in the film Nightwatch -LRB- 1994 -RRB- ."),
              (3,
               'Since then he has appeared in numerous films in his native Scandinavia and Europe in general , including Headhunters -LRB- 2011 -RRB- and A Thousand Times Good Night -LRB- 2013 -RRB- .'),
              (7,
               'He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot .'),
              (8,
               'He became widely known to a broad audience for his current role as Ser Jaime Lannister , in the HBO series Game o

In [30]:
# Check if the page exists in FEVER_df without normalization
page_to_check = "José_María_Chacón"
any(page_to_check in str(group) for group in FEVER_df['golden_evidence_group'])

False

In [ ]:
# Checking for the same page name but this after normalization in FEVER_df
page_to_check = "José_María_Chacón"
exists = norm_page_exists_in_df(FEVER_df, page_to_check)
print(exists)

True


The above code returns true means the page is there but we are not able to find it because of the unicode mis-match. Therefore, page name in FEVER_df['golden_evidence_group'] also needs to be normalized.

## 4.8 Normalize FEVER_df

In [31]:
# Normalize page names inside golden_evidence_group
for row in FEVER_df.itertuples(index=False):
    group = row.golden_evidence_group
    if not group:
        continue
    for item in group:
        item[2] = normalize(item[2])

In [32]:
# Check if the page exists in FEVER_df
page_to_check = "José_María_Chacón"
any(page_to_check in str(group) for group in FEVER_df['golden_evidence_group'])

True

This indicates that normalization done successfully

## 4.9 Adding sentence text to 'FEVER_df'

In [77]:
FEVER_df.head()

,id,label,claim,evidence,golden_evidence_group,golden_evidence_text
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [...","[[92206, 104971, Nikolaj_Coster-Waldau, 7], [9...",
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271...","[[174271, 187498, Roman_Atwood, 1]]",
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]","[[255136, 254645, History_of_art, 2]]",
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]","[[180804, 193183, Adrienne_Bailon, 0]]",
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]","[[100277, None, None, None]]",EMPTY


In [ ]:
# Adding sentence text to the DataFrame

# To see the progress
tqdm.pandas()

# Apply to all rows
FEVER_df["golden_evidence_text"] = FEVER_df["golden_evidence_group"].progress_apply(
    lambda group: extract_evidence_text(group, enriched_fever_index)
)

FEVER_df.head()

100%|██████████| 165447/165447 [00:00<00:00, 745260.35it/s]


,id,label,claim,evidence,golden_evidence_group,golden_evidence_text
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [...","[[92206, 104971, Nikolaj_Coster-Waldau, 7], [9...",He then played Detective John Amsterdam in the...
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271...","[[174271, 187498, Roman_Atwood, 1]]","He is best known for his vlogs , where he post..."
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]","[[255136, 254645, History_of_art, 2]]",The subsequent expansion of the list of princi...
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]","[[180804, 193183, Adrienne_Bailon, 0]]",Adrienne Eliza Houghton -LRB- née Bailon ; bor...
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]","[[100277, None, None, None]]",


### 4.9.1 Verifying Created Dataframe

In this, we 'll check whether the sentence text has been added succesfully or not. For this, we visualize & analyse the data in 3 different parts as per the labels - 'SUPPORTS', 'REFUTES' & 'NOT ENOUGH INFO'

For each type of data, we do the following steps
1. Filter the data with the label type and use 'head()' to visualize first few entires.
2. We 'll count the no. of entries using the 'shape' on this filtered data
3. We 'll count no. of empty strings in the 'golden_evidence_text'.

#### 4.9.1.1 Verifying where label = 'NOT ENOUGH INFO'

In [81]:
FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO'].head()

,id,label,claim,evidence,golden_evidence_group,golden_evidence_text
4,83235,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]","[[100277, None, None, None]]",
6,149579,NOT ENOUGH INFO,Beautiful reached number two on the Billboard ...,"[[[173384, None, None, None]]]","[[173384, None, None, None]]",
7,229289,NOT ENOUGH INFO,Neal Schon was named in 1954.,"[[[273626, None, None, None]]]","[[273626, None, None, None]]",
21,138117,NOT ENOUGH INFO,John Wick: Chapter 2 was theatrically released...,"[[[160895, None, None, None]]]","[[160895, None, None, None]]",
23,210010,NOT ENOUGH INFO,Afghanistan is the source of the Kushan dynasty.,"[[[248748, None, None, None]]]","[[248748, None, None, None]]",


In [90]:
FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO'].shape

(42305, 6)

In [96]:
# Counting the no. of entries with empty 'golden_evidence_text'
print((FEVER_df[FEVER_df['label'] == 'NOT ENOUGH INFO']['golden_evidence_text']=='').sum())

42305


This is as expetced. As for such entries, we are entring empty string.

#### 4.9.1.2 Verifying where label = 'SUPPORTS'

In [82]:
FEVER_df[FEVER_df['label'] == 'SUPPORTS'].head()

,id,label,claim,evidence,golden_evidence_group,golden_evidence_text
0,75397,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [...","[[92206, 104971, Nikolaj_Coster-Waldau, 7], [9...",He then played Detective John Amsterdam in the...
1,150448,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271...","[[174271, 187498, Roman_Atwood, 1]]","He is best known for his vlogs , where he post..."
2,214861,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]","[[255136, 254645, History_of_art, 2]]",The subsequent expansion of the list of princi...
5,129629,SUPPORTS,Homeland is an American television spy thrille...,"[[[151831, 166598, Homeland_-LRB-TV_series-RRB...","[[151831, 166598, Homeland_-LRB-TV_series-RRB-...",Homeland is an American spy thriller televisio...
8,33078,SUPPORTS,The Boston Celtics play their home games at TD...,"[[[49158, 58489, Boston_Celtics, 3]], [[49159,...","[[49158, 58489, Boston_Celtics, 3]]",The Celtics play their home games at the TD Ga...


In [93]:
FEVER_df[FEVER_df['label'] == 'SUPPORTS'].shape

(86701, 6)

In [97]:
# Counting the no. of entries with empty 'golden_evidence_text'
print((FEVER_df[FEVER_df['label'] == 'SUPPORTS']['golden_evidence_text']=='').sum())

0


This is as exptected

#### 4.9.1.1 Verifying where label = 'REFUTES'

In [83]:
FEVER_df[FEVER_df['label'] == 'REFUTES'].head()

,id,label,claim,evidence,golden_evidence_group,golden_evidence_text
3,156709,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]","[[180804, 193183, Adrienne_Bailon, 0]]",Adrienne Eliza Houghton -LRB- née Bailon ; bor...
14,138503,REFUTES,"Stranger Things is set in Bloomington, Indiana.","[[[161295, 175782, Stranger_Things, 5]]]","[[161295, 175782, Stranger_Things, 5]]","Set in the fictional town of Hawkins , Indiana..."
16,73170,REFUTES,Puerto Rico is not an unincorporated territory...,"[[[89957, 102650, Puerto_Rico, 0]]]","[[89957, 102650, Puerto_Rico, 0]]",Puerto Rico -LRB- Spanish for `` Rich Port '' ...
26,15812,REFUTES,Peggy Sue Got Married is a Egyptian film relea...,"[[[31205, 37902, Peggy_Sue_Got_Married, 0], [3...","[[31205, 37902, Peggy_Sue_Got_Married, 0], [31...",Peggy Sue Got Married is a 1986 American comed...
27,57330,REFUTES,Andy Roddick lost 5 Master Series between 2002...,"[[[73660, 84912, Andy_Roddick, 7]]]","[[73660, 84912, Andy_Roddick, 7]]",Roddick was ranked in the top 10 for nine cons...


In [99]:
FEVER_df[FEVER_df['label'] == 'REFUTES'].shape

(36441, 6)

In [98]:
# Counting the no. of entries with empty 'golden_evidence_text'
print((FEVER_df[FEVER_df['label'] == 'REFUTES']['golden_evidence_text']=='').sum())

0


This is as expected

## 4.10 Filtering the required data

Since, we only need 'label', 'claim' & 'golden_evidence_text' for our experiment, we can drop rest of the columns.

In [104]:
FEVER_df = FEVER_df[["label", "claim", "golden_evidence_text"]]
FEVER_df

,label,claim,golden_evidence_text
0,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,He then played Detective John Amsterdam in the...
1,SUPPORTS,Roman Atwood is a content creator.,"He is best known for his vlogs , where he post..."
2,SUPPORTS,"History of art includes architecture, dance, s...",The subsequent expansion of the list of princi...
3,REFUTES,Adrienne Bailon is an accountant.,Adrienne Eliza Houghton -LRB- née Bailon ; bor...
4,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,
...,...,...,...
165442,NOT ENOUGH INFO,Heavy Metal music was first developed in the U...,
165443,NOT ENOUGH INFO,Bethany Hamilton was a philosopher.,
165444,NOT ENOUGH INFO,Mike Huckabee hosts Oprah.,
165445,NOT ENOUGH INFO,Excuse My French is a studio EP.,


## 4.11 Saving the final Dataframe

In [105]:
# Saving the final DataFrame to a JSONL file
'''
'force_ascii=False' 'll help us to keep the diacritics in the text as it is, without
 converting them to escape sequences i.e.

 If not 'False', then, names like this in our dataset 'll be converted to

 "José" → "Jos\u00e9"
"Mārīa" → "M\u0101r\u012ba"

Where as if it is 'False', then, names like this in our dataset 'll be kept as it is
"Mārīa" → "Mārīa"
"José" → "José"
'''
output_path = os.path.join("..", "datasets", "FEVER", "pre-processed", "Final_FEVER_Dataset_Eng.jsonl")
FEVER_df.to_json(output_path, orient="records", lines=True, force_ascii=False)

## 4.12 Verifying the saved data

In [108]:
Final_FEVER_df = get_dataset_dataframe("FEVER", "pre-processed", "Final_FEVER_Dataset_Eng.jsonl")
Final_FEVER_df.head()

,label,claim,golden_evidence_text
0,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,He then played Detective John Amsterdam in the...
1,SUPPORTS,Roman Atwood is a content creator.,"He is best known for his vlogs , where he post..."
2,SUPPORTS,"History of art includes architecture, dance, s...",The subsequent expansion of the list of princi...
3,REFUTES,Adrienne Bailon is an accountant.,Adrienne Eliza Houghton -LRB- née Bailon ; bor...
4,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,


# IMPORTANT NOTE - Need to do this later

Need to check for duplicates in the data. The below code I have tried in the previous notebook.

In [111]:
# Check for duplicate claims in FEVER_df
duplicate_claims = Final_FEVER_df[Final_FEVER_df.duplicated(keep=False)]
print(f"Number of duplicate claims: {duplicate_claims.shape[0]}")
duplicate_claims

Number of duplicate claims: 13468


,label,claim,golden_evidence_text
2,SUPPORTS,"History of art includes architecture, dance, s...",The subsequent expansion of the list of princi...
7,NOT ENOUGH INFO,Neal Schon was named in 1954.,
18,SUPPORTS,Stranger than Fiction is a film.,Stranger than Fiction is a 2006 American fanta...
24,NOT ENOUGH INFO,Marilyn Monroe worked with Warner Brothers.,
31,SUPPORTS,The Jim Henson Company produced The Muppet Mov...,"The company has also produced many films , inc..."
...,...,...,...
165313,SUPPORTS,Lockhead Martin F-35 Lightning II was an expen...,The program is the most expensive military wea...
165360,NOT ENOUGH INFO,Polynesian languages includes several languages.,
165377,NOT ENOUGH INFO,Noel Fisher starred in Game of Thrones.,
165382,NOT ENOUGH INFO,Edmund H. North won an Emmy Award.,


# END